In [ ]:
########################################################################
# Inclusão das Bibliotecas Necessárias
########################################################################
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
########################################################################
# Localizando o Diretório Base
########################################################################
%cd /content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código


/content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código


In [ ]:
# =============================

In [1]:
"""
IMPLEMENTAÇÃO HATRPO OTIMIZADA - VERSÃO 6.1.0
- Parâmetros otimizados para melhor desempenho
- Aumento da capacidade das redes
- Melhor exploração
- Recompensas ajustadas
"""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from collections import deque
import random
from typing import List, Tuple, Dict, Optional
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pandas as pd
from datetime import datetime
import os
from pathlib import Path
import warnings
import time
import gc
import math

warnings.filterwarnings('ignore')

# ==================== CONFIGURAÇÃO OTIMIZADA ====================
MAP_CONFIG = {
    'height': 12,
    'width': 8,
    'grid': [
        ['R1', '0', '0', '0', '0', '0', '0', 'R2'],
        ['0', 'Y', 'A', 'A', 'A', 'A', '0', '0'],
        ['0', '0', 'A', 'A', 'A', 'A', '0', '0'],
        ['X', '0', '0', '0', 'X', '0', 'Y', '0'],
        ['0', 'Y', 'X', '0', '0', '0', '0', '0'],
        ['0', '0', '0', 'X', '0', 'Y', 'X', 'X'],
        ['0', 'X', '0', 'Y', '0', 'X', '0', '0'],
        ['0', '0', '0', 'X', '0', '0', '0', '0'],
        ['X', '0', 'Y', '0', '0', '0', 'X', '0'],
        ['X', '0', 'B', 'B', 'B', 'B', 'X', '0'],
        ['X', '0', 'B', 'B', 'B', 'B', 'Y', '0'],
        ['0', '0', '0', 'Y', '0', '0', '0', '0']
    ]
}

class HATRPOConfigOptimized:
    # ========== EXPLORAÇÃO (AUMENTADA) ==========
    EPSILON_START = 1.0
    EPSILON_END = 0.01           # Reduzido para mais explotação no final
    EPSILON_DECAY_STEPS = 150000  # Aumentado para explorar mais tempo

    # ========== APRENDIZADO (OTIMIZADO) ==========
    ACTOR_LR = 3e-4              # Aumentado (antes 1e-4)
    CRITIC_LR = 3e-3             # Aumentado (antes 1e-3)
    GAMMA = 0.99
    LAMBDA = 0.95
    MAX_KL = 0.02                # Aumentado para permitir mudanças maiores
    BATCH_SIZE = 256             # Aumentado (antes 128)

    # ========== REDES NEURAIS (MAIOR CAPACIDADE) ==========
    HIDDEN_DIM = 512             # Aumentado (antes 256)
    DROPOUT_RATE = 0.1
    NUM_LAYERS = 3               # Camadas adicionais

    # ========== MEMÓRIA ==========
    BUFFER_SIZE = 500000         # Aumentado (antes 200000)
    LEARNING_STARTS = 5000       # Aumentado (antes 2000)

    # ========== TREINAMENTO ==========
    MAX_GRAD_NORM = 1.0
    MAX_STEPS = 500
    EPISODES_TOTAL = 3000

    # ========== REWARDS (AJUSTADOS PARA INCENTIVAR) ==========
    MOVE_PENALTY = -0.005        # Penalidade menor (antes -0.01)
    PICKUP_REWARD = 10.0         # Aumentado (antes 5.0)
    DROP_REWARD = 50.0           # Aumentado (antes 25.0)
    DISTANCE_REWARD = 0.5        # Aumentado (antes 0.2)
    COMPLETION_BONUS = 100.0     # Aumentado (antes 50.0)
    COLLISION_PENALTY = -0.5     # Aumentado (antes -0.3)

    # ========== SISTEMA ==========
    SAVE_INTERVAL = 100
    CLEAN_MEMORY_EVERY = 500

    # ========== FALHAS ==========
    FAILURE_PROBABILITY = 0.15   # Reduzido (antes 0.2)
    FAILURE_PENALTY = -0.1       # Penalidade menor (antes -0.15)
    ALTERNATIVE_DIRECTIONS = True

    # ========== NOVAS OTIMIZAÇÕES ==========
    USE_GRADIENT_CLIPPING = True
    TARGET_UPDATE_FREQ = 100
    ENTROPY_COEF = 0.01          # Coeficiente de entropia para exploração


# ==================== REDE NEURAL MELHORADA ====================
class ImprovedActorNetwork(nn.Module):
    """Rede de política melhorada com mais camadas e residual connections"""
    def __init__(self, state_dim, action_dim, hidden_dim=512, num_layers=3, dropout=0.1):
        super().__init__()

        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        # Camada de entrada
        self.input_layer = nn.Linear(state_dim, hidden_dim)

        # Camadas escondidas com residual connections
        self.hidden_layers = nn.ModuleList()
        for i in range(num_layers - 1):
            self.hidden_layers.append(nn.Linear(hidden_dim, hidden_dim))

        # Camada de saída
        self.output_layer = nn.Linear(hidden_dim, action_dim)

        # Layer norms para estabilidade
        self.layer_norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])

        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU()

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.orthogonal_(module.weight, gain=0.01)
            nn.init.constant_(module.bias, 0.0)

    def forward(self, state):
        x = self.activation(self.input_layer(state))
        x = self.layer_norms[0](x)
        x = self.dropout(x)

        for i, layer in enumerate(self.hidden_layers):
            residual = x
            x = self.activation(layer(x))
            x = self.layer_norms[i + 1](x)
            x = self.dropout(x)
            x = x + residual  # Residual connection

        logits = self.output_layer(x)
        return F.softmax(logits, dim=-1), logits


class ImprovedCriticNetwork(nn.Module):
    """Rede de valor melhorada com mais capacidade"""
    def __init__(self, global_state_dim, hidden_dim=512, num_layers=3, dropout=0.1):
        super().__init__()

        self.num_layers = num_layers

        self.input_layer = nn.Linear(global_state_dim, hidden_dim)

        self.hidden_layers = nn.ModuleList()
        for i in range(num_layers - 1):
            self.hidden_layers.append(nn.Linear(hidden_dim, hidden_dim))

        self.output_layer = nn.Linear(hidden_dim, 1)

        self.layer_norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])

        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU()

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.orthogonal_(module.weight, gain=0.01)
            nn.init.constant_(module.bias, 0.0)

    def forward(self, global_state):
        x = self.activation(self.input_layer(global_state))
        x = self.layer_norms[0](x)
        x = self.dropout(x)

        for i, layer in enumerate(self.hidden_layers):
            x = self.activation(layer(x))
            x = self.layer_norms[i + 1](x)
            x = self.dropout(x)

        return self.output_layer(x)


# ==================== AGENTE HATRPO OTIMIZADO ====================
class HATRPOAgentOptimized:
    def __init__(self, agent_id, state_dim, action_dim, global_state_dim, config):
        self.agent_id = agent_id
        self.action_dim = action_dim
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"  Agente {agent_id} usando dispositivo: {self.device}")

        self.actor = ImprovedActorNetwork(state_dim, action_dim,
                                          config.HIDDEN_DIM,
                                          config.NUM_LAYERS,
                                          config.DROPOUT_RATE).to(self.device)
        self.actor_old = ImprovedActorNetwork(state_dim, action_dim,
                                              config.HIDDEN_DIM,
                                              config.NUM_LAYERS,
                                              config.DROPOUT_RATE).to(self.device)
        self.actor_old.load_state_dict(self.actor.state_dict())

        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=config.ACTOR_LR)

        self.steps_done = 0
        self.epsilon = config.EPSILON_START
        self.total_episodes = 0

    def get_epsilon(self):
        if self.steps_done >= self.config.EPSILON_DECAY_STEPS:
            return self.config.EPSILON_END
        epsilon = self.config.EPSILON_START - (self.config.EPSILON_START - self.config.EPSILON_END) * \
                  self.steps_done / self.config.EPSILON_DECAY_STEPS
        return max(self.config.EPSILON_END, epsilon)

    def select_action(self, state, training=True):
        self.steps_done += 1

        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            probs, _ = self.actor(state_tensor)

            if training and random.random() < self.get_epsilon():
                action = random.randrange(self.action_dim)
                log_prob = torch.log(probs[0, action] + 1e-10).item()
            else:
                action = probs.argmax().item()
                log_prob = torch.log(probs[0, action] + 1e-10).item()

            return action, log_prob

    def get_action_probs(self, states):
        with torch.no_grad():
            states_tensor = torch.FloatTensor(states).to(self.device)
            probs, _ = self.actor(states_tensor)
            return probs

    def get_old_action_probs(self, states):
        with torch.no_grad():
            states_tensor = torch.FloatTensor(states).to(self.device)
            probs, _ = self.actor_old(states_tensor)
            return probs

    def update_actor(self, states, actions, advantages, old_log_probs):
        """Atualiza o ator usando PPO com clipping"""
        states_tensor = torch.FloatTensor(states).to(self.device)
        actions_tensor = torch.LongTensor(actions).to(self.device)
        advantages_tensor = torch.FloatTensor(advantages).to(self.device)
        old_log_probs_tensor = torch.FloatTensor(old_log_probs).to(self.device)

        # Calcular novas probabilidades
        probs, _ = self.actor(states_tensor)
        new_log_probs = torch.log(probs.gather(1, actions_tensor.unsqueeze(1)).squeeze() + 1e-10)

        # Calcular ratio e loss PPO
        ratio = torch.exp(new_log_probs - old_log_probs_tensor)
        surr1 = ratio * advantages_tensor
        surr2 = torch.clamp(ratio, 0.8, 1.2) * advantages_tensor
        policy_loss = -torch.min(surr1, surr2).mean()

        # Entropy bonus para exploração
        entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=1).mean()
        total_loss = policy_loss - self.config.ENTROPY_COEF * entropy

        self.actor_optimizer.zero_grad()
        total_loss.backward()

        if self.config.USE_GRADIENT_CLIPPING:
            torch.nn.utils.clip_grad_norm_(self.actor.parameters(), self.config.MAX_GRAD_NORM)

        self.actor_optimizer.step()

        # Atualizar política antiga periodicamente
        if self.steps_done % self.config.TARGET_UPDATE_FREQ == 0:
            self.actor_old.load_state_dict(self.actor.state_dict())

        return total_loss.item()

    def save_model(self, save_dir, episode):
        os.makedirs(save_dir, exist_ok=True)
        torch.save(self.actor.state_dict(), save_dir / f"hatrpo_actor_{self.agent_id}_ep{episode}.pth")


# ==================== CRÍTICO CENTRALIZADO OTIMIZADO ====================
class CentralizedCriticOptimized:
    def __init__(self, global_state_dim, config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.critic = ImprovedCriticNetwork(global_state_dim,
                                            config.HIDDEN_DIM,
                                            config.NUM_LAYERS,
                                            config.DROPOUT_RATE).to(self.device)
        self.critic_target = ImprovedCriticNetwork(global_state_dim,
                                                   config.HIDDEN_DIM,
                                                   config.NUM_LAYERS,
                                                   config.DROPOUT_RATE).to(self.device)
        self.critic_target.load_state_dict(self.critic.state_dict())

        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=config.CRITIC_LR)

    def get_value(self, global_state):
        with torch.no_grad():
            state_tensor = torch.FloatTensor(global_state).unsqueeze(0).to(self.device)
            return self.critic(state_tensor).item()

    def get_values(self, global_states):
        with torch.no_grad():
            states_tensor = torch.FloatTensor(global_states).to(self.device)
            return self.critic(states_tensor).squeeze().cpu().numpy()

    def update(self, global_states, returns):
        states_tensor = torch.FloatTensor(global_states).to(self.device)
        returns_tensor = torch.FloatTensor(returns).to(self.device)

        values = self.critic(states_tensor).squeeze()
        critic_loss = F.mse_loss(values, returns_tensor)

        self.critic_optimizer.zero_grad()
        critic_loss.backward()

        if self.config.USE_GRADIENT_CLIPPING:
            torch.nn.utils.clip_grad_norm_(self.critic.parameters(), self.config.MAX_GRAD_NORM)

        self.critic_optimizer.step()

        # Soft update target network
        for target_param, param in zip(self.critic_target.parameters(), self.critic.parameters()):
            target_param.data.copy_(0.995 * target_param.data + 0.005 * param.data)

        return critic_loss.item()

    def compute_gae(self, rewards, values, dones):
        advantages = []
        gae = 0

        for t in reversed(range(len(rewards))):
            if t == len(rewards) - 1:
                next_value = 0
            else:
                next_value = values[t + 1]

            delta = rewards[t] + self.config.GAMMA * next_value * (1 - dones[t]) - values[t]
            gae = delta + self.config.GAMMA * self.config.LAMBDA * (1 - dones[t]) * gae
            advantages.insert(0, gae)

        returns = np.array(advantages) + np.array(values)
        return np.array(advantages), returns

    def save_model(self, save_dir, episode):
        os.makedirs(save_dir, exist_ok=True)
        torch.save(self.critic.state_dict(), save_dir / f"hatrpo_critic_ep{episode}.pth")


# ==================== TRAJECTORY BUFFER ====================
class TrajectoryBuffer:
    def __init__(self):
        self.states = []
        self.actions = []
        self.rewards = []
        self.next_states = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.global_states = []
        self.next_global_states = []

    def push(self, state, action, reward, next_state, done, log_prob, value, global_state, next_global_state):
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.next_states.append(next_state)
        self.dones.append(done)
        self.log_probs.append(log_prob)
        self.values.append(value)
        self.global_states.append(global_state)
        self.next_global_states.append(next_global_state)

    def clear(self):
        self.states = []
        self.actions = []
        self.rewards = []
        self.next_states = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.global_states = []
        self.next_global_states = []

    def get_trajectory(self):
        return {
            'states': np.array(self.states),
            'actions': np.array(self.actions),
            'rewards': np.array(self.rewards),
            'next_states': np.array(self.next_states),
            'dones': np.array(self.dones),
            'log_probs': np.array(self.log_probs),
            'values': np.array(self.values),
            'global_states': np.array(self.global_states),
            'next_global_states': np.array(self.next_global_states)
        }

    def __len__(self):
        return len(self.states)


# ==================== AMBIENTE WAREHOUSE ====================
class WarehouseEnv(gym.Env):
    def __init__(self, config=None):
        super().__init__()

        self.config = config or HATRPOConfigOptimized()
        self.height = MAP_CONFIG['height']
        self.width = MAP_CONFIG['width']
        self.base_grid = [row[:] for row in MAP_CONFIG['grid']]
        self.grid = [row[:] for row in self.base_grid]

        self.y_positions = self._find_positions('Y')

        self.robot_positions = None
        self.box_positions = None
        self.targets = self._find_positions('B')

        self.num_robots = 2
        self.num_boxes = len(self._find_positions('A'))
        self.num_targets = len(self.targets)

        self.delivered_boxes = None
        self.steps = 0
        self.max_steps = self.config.MAX_STEPS
        self.total_deliveries = 0
        self.collisions = 0
        self.failures = [0, 0]
        self.distance_traveled = [0, 0]
        self.previous_distances = None

        self.action_space = spaces.Tuple([spaces.Discrete(6) for _ in range(self.num_robots)])

        self.state_dim = (self.num_robots * 2) + (self.num_boxes * 2) + (self.num_targets * 2) + 8
        self.observation_space = spaces.Box(
            low=-1, high=self.height + self.width,
            shape=(self.state_dim,),
            dtype=np.float32
        )

        self.global_state_dim = 30

    def _find_positions(self, symbols):
        if isinstance(symbols, str):
            symbols = [symbols]
        positions = []
        for i in range(self.height):
            for j in range(self.width):
                cell = self.grid[i][j]
                if any(cell.startswith(sym) for sym in symbols):
                    positions.append((i, j))
        return positions

    def _remove_random_y_barriers(self):
        self.grid = [row[:] for row in self.base_grid]
        for y_pos in self.y_positions:
            if random.random() < 0.5:
                i, j = y_pos
                self.grid[i][j] = '0'

    def reset(self):
        self._remove_random_y_barriers()

        self.robot_positions = self._find_positions('R')
        self.box_positions = self._find_positions('A')
        self.delivered_boxes = [False] * self.num_boxes
        self.targets = self._find_positions('B')

        self.steps = 0
        self.total_deliveries = 0
        self.collisions = 0
        self.failures = [0, 0]
        self.distance_traveled = [0, 0]

        self.previous_distances = [self._min_distance_to_boxes(r) for r in range(self.num_robots)]

        all_states = self._get_all_states()
        return all_states, self._get_global_state()

    def _get_all_states(self):
        all_states = []
        for robot_id in range(self.num_robots):
            state = self._get_observation_for_robot(robot_id)
            all_states.append(state)
        return np.array(all_states, dtype=np.float32)

    def _get_observation_for_robot(self, robot_id):
        obs = []

        robot_pos = self.robot_positions[robot_id]
        obs.append(robot_pos[0] / self.height)
        obs.append(robot_pos[1] / self.width)

        for other_id, other_pos in enumerate(self.robot_positions):
            if other_id != robot_id:
                obs.append(other_pos[0] / self.height)
                obs.append(other_pos[1] / self.width)

        for box_id, box_pos in enumerate(self.box_positions):
            if box_pos is None or self.delivered_boxes[box_id]:
                obs.append(-1)
                obs.append(-1)
            else:
                obs.append(box_pos[0] / self.height)
                obs.append(box_pos[1] / self.width)

        for target_pos in self.targets:
            obs.append(target_pos[0] / self.height)
            obs.append(target_pos[1] / self.width)

        min_box_dist = min([abs(robot_pos[0] - box_pos[0]) + abs(robot_pos[1] - box_pos[1])
                           for box_pos in self.box_positions
                           if box_pos is not None and
                           not self.delivered_boxes[self.box_positions.index(box_pos)]],
                          default=100)
        obs.append(min_box_dist / (self.height + self.width))

        min_target_dist = min([abs(robot_pos[0] - target_pos[0]) + abs(robot_pos[1] - target_pos[1])
                              for target_pos in self.targets],
                             default=100)
        obs.append(min_target_dist / (self.height + self.width))

        return np.array(obs, dtype=np.float32)

    def _min_distance_to_boxes(self, robot_id):
        robot_pos = self.robot_positions[robot_id]
        remaining_boxes = [box_pos for box_pos in self.box_positions
                          if box_pos is not None and
                          not self.delivered_boxes[self.box_positions.index(box_pos)]]

        if not remaining_boxes:
            return 0
        return min([abs(robot_pos[0] - box_pos[0]) + abs(robot_pos[1] - box_pos[1])
                   for box_pos in remaining_boxes])

    def _is_valid_position(self, pos, robot_id=None):
        i, j = pos
        if i < 0 or i >= self.height or j < 0 or j >= self.width:
            return False

        cell = self.grid[i][j]
        if cell in ['X', 'Y']:
            return False

        if robot_id is not None:
            for rid, rpos in enumerate(self.robot_positions):
                if rid != robot_id and rpos == (i, j):
                    return False
        return True

    def _get_alternative_direction(self, original_action, robot_id):
        i, j = self.robot_positions[robot_id]
        alternative_actions = [a for a in range(4) if a != original_action]
        random.shuffle(alternative_actions)

        for alt_action in alternative_actions:
            alt_i, alt_j = i, j
            if alt_action == 0: alt_i -= 1
            elif alt_action == 1: alt_i += 1
            elif alt_action == 2: alt_j -= 1
            elif alt_action == 3: alt_j += 1

            if self._is_valid_position((alt_i, alt_j), robot_id):
                return alt_action, (alt_i, alt_j)
        return None, None

    def _move_robot_with_failure(self, robot_id, action):
        i, j = self.robot_positions[robot_id]
        new_i, new_j = i, j

        if action == 0: new_i -= 1
        elif action == 1: new_i += 1
        elif action == 2: new_j -= 1
        elif action == 3: new_j += 1

        desired_valid = self._is_valid_position((new_i, new_j), robot_id)

        if random.random() < self.config.FAILURE_PROBABILITY:
            self.failures[robot_id] += 1

            if self.config.ALTERNATIVE_DIRECTIONS:
                alt_action, alt_pos = self._get_alternative_direction(action, robot_id)
                if alt_action is not None:
                    old_pos = self.robot_positions[robot_id]
                    self.grid[old_pos[0]][old_pos[1]] = '0'
                    self.grid[alt_pos[0]][alt_pos[1]] = f'R{robot_id + 1}'
                    self.robot_positions[robot_id] = alt_pos
                    self.distance_traveled[robot_id] += 1
                    return self.config.FAILURE_PENALTY - 0.01
            return self.config.FAILURE_PENALTY

        if desired_valid:
            self.distance_traveled[robot_id] += 1
            old_pos = self.robot_positions[robot_id]
            self.grid[old_pos[0]][old_pos[1]] = '0'
            self.grid[new_i][new_j] = f'R{robot_id + 1}'
            self.robot_positions[robot_id] = (new_i, new_j)
            return self.config.MOVE_PENALTY
        else:
            self.collisions += 1
            return self.config.COLLISION_PENALTY

    def _pickup_box(self, robot_id):
        robot_pos = self.robot_positions[robot_id]
        for box_id, box_pos in enumerate(self.box_positions):
            if not self.delivered_boxes[box_id] and box_pos == robot_pos:
                self.box_positions[box_id] = None
                self.grid[robot_pos[0]][robot_pos[1]] = f'R{robot_id + 1}'
                return self.config.PICKUP_REWARD
        return -self.config.MOVE_PENALTY

    def _drop_box(self, robot_id):
        robot_pos = self.robot_positions[robot_id]

        box_with_robot = None
        for box_id, box_pos in enumerate(self.box_positions):
            if box_pos is None and not self.delivered_boxes[box_id]:
                box_with_robot = box_id
                break

        if box_with_robot is None:
            return -self.config.MOVE_PENALTY

        for target_pos in self.targets:
            if robot_pos == target_pos:
                self.delivered_boxes[box_with_robot] = True
                self.total_deliveries += 1
                self.grid[robot_pos[0]][robot_pos[1]] = f'R{robot_id + 1}'
                return self.config.DROP_REWARD
        return -self.config.MOVE_PENALTY * 2

    def _calculate_shaped_reward(self, robot_id, base_reward):
        reward = base_reward

        current_distance = self._min_distance_to_boxes(robot_id)
        previous_distance = self.previous_distances[robot_id]

        if current_distance < previous_distance:
            reward += self.config.DISTANCE_REWARD * (previous_distance - current_distance)
        elif current_distance > previous_distance:
            reward -= self.config.DISTANCE_REWARD * 0.2 * (current_distance - previous_distance)

        self.previous_distances[robot_id] = current_distance

        if all(self.delivered_boxes):
            reward += self.config.COMPLETION_BONUS

        return reward

    def _get_global_state(self):
        global_state = []

        for robot_pos in self.robot_positions:
            global_state.append(robot_pos[0] / self.height)
            global_state.append(robot_pos[1] / self.width)

        for box_pos in self.box_positions:
            if box_pos is not None:
                global_state.append(box_pos[0] / self.height)
                global_state.append(box_pos[1] / self.width)
            else:
                global_state.append(-1)
                global_state.append(-1)

        for delivered in self.delivered_boxes:
            global_state.append(1.0 if delivered else 0.0)

        global_state.append(self.steps / self.max_steps)
        global_state.append(self.total_deliveries / self.num_boxes)

        return np.array(global_state, dtype=np.float32)

    def step(self, actions):
        self.steps += 1

        if len(actions) != self.num_robots:
            actions = [actions] * self.num_robots

        total_reward = 0
        rewards = [0, 0]

        movement_rewards = []
        for robot_id, action in enumerate(actions):
            if action < 4:
                reward = self._move_robot_with_failure(robot_id, action)
                movement_rewards.append(reward)
            else:
                movement_rewards.append(0)

        interaction_rewards = []
        for robot_id, action in enumerate(actions):
            if action == 4:
                reward = self._pickup_box(robot_id)
                interaction_rewards.append(reward)
            elif action == 5:
                reward = self._drop_box(robot_id)
                interaction_rewards.append(reward)
            else:
                interaction_rewards.append(0)

        for robot_id in range(self.num_robots):
            base_reward = movement_rewards[robot_id] + interaction_rewards[robot_id]
            shaped_reward = self._calculate_shaped_reward(robot_id, base_reward)
            rewards[robot_id] = shaped_reward
            total_reward += shaped_reward

        terminated = all(self.delivered_boxes)
        truncated = self.steps >= self.max_steps

        next_all_states = self._get_all_states()
        next_global_state = self._get_global_state()
        info = self._get_info()

        return next_all_states, rewards, terminated, truncated, info, next_all_states, next_global_state

    def _get_info(self):
        return {
            'steps': self.steps,
            'total_deliveries': self.total_deliveries,
            'collisions': self.collisions,
            'failures_r1': self.failures[0],
            'failures_r2': self.failures[1],
            'distance_traveled': self.distance_traveled.copy(),
            'remaining_boxes': sum(1 for d in self.delivered_boxes if not d),
            'success_rate': self.total_deliveries / self.num_boxes if self.steps > 0 else 0,
        }

    def close(self):
        pass


# ==================== FUNÇÕES DE PLOTAGEM ====================
def plot_individual_graphs(metrics, save_dir):
    print(f"\n📊 Gerando gráficos individuais em: {save_dir}")

    episodes = range(1, len(metrics['episode_rewards']) + 1)

    # Entregas
    plt.figure(figsize=(14, 7))
    plt.plot(episodes, metrics['episode_deliveries'], 'g-', alpha=0.5, linewidth=1)
    if len(metrics['episode_deliveries']) >= 100:
        moving_avg_del = np.convolve(metrics['episode_deliveries'], np.ones(100)/100, mode='valid')
        plt.plot(range(100, len(metrics['episode_deliveries']) + 1), moving_avg_del, 'r-', linewidth=2, label='Média Móvel (100)')
    plt.axhline(y=8, color='orange', linestyle='--', linewidth=2, label='Meta (8 caixas)')
    plt.xlabel('Episódio', fontsize=12)
    plt.ylabel('Entregas', fontsize=12)
    plt.title('Entregas por Episódio - HATRPO Otimizado (3000 episódios)', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim([0, 9])
    plt.tight_layout()
    plt.savefig(save_dir / 'grafico_entregas_hatrpo.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ grafico_entregas_hatrpo.png salvo")

    # Recompensa
    plt.figure(figsize=(14, 7))
    plt.plot(episodes, metrics['episode_rewards'], 'b-', alpha=0.5, linewidth=1)
    if len(metrics['episode_rewards']) >= 100:
        moving_avg = np.convolve(metrics['episode_rewards'], np.ones(100)/100, mode='valid')
        plt.plot(range(100, len(metrics['episode_rewards']) + 1), moving_avg, 'r-', linewidth=2, label='Média Móvel (100)')
    plt.xlabel('Episódio', fontsize=12)
    plt.ylabel('Recompensa', fontsize=12)
    plt.title('Recompensa por Episódio - HATRPO Otimizado (3000 episódios)', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / 'grafico_recompensa_hatrpo.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ grafico_recompensa_hatrpo.png salvo")

    # Taxa de Sucesso
    plt.figure(figsize=(14, 7))
    plt.plot(episodes, metrics['success_rates'], 'purple', alpha=0.5, linewidth=1)
    if len(metrics['success_rates']) >= 100:
        moving_avg_success = np.convolve(metrics['success_rates'], np.ones(100)/100, mode='valid')
        plt.plot(range(100, len(metrics['success_rates']) + 1), moving_avg_success, 'r-', linewidth=2, label='Média Móvel (100)')
    plt.axhline(y=0.95, color='green', linestyle='--', linewidth=2, label='Meta 95%')
    plt.xlabel('Episódio', fontsize=12)
    plt.ylabel('Taxa de Sucesso', fontsize=12)
    plt.title('Taxa de Sucesso por Episódio - HATRPO Otimizado (3000 episódios)', fontsize=14, fontweight='bold')
    plt.ylim([0, 1.05])
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / 'grafico_taxa_sucesso_hatrpo.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ grafico_taxa_sucesso_hatrpo.png salvo")

    print(f"📊 Todos os gráficos foram salvos em: {save_dir}")


def save_metrics_csv(metrics, save_dir):
    df = pd.DataFrame({
        'episodio': range(1, len(metrics['episode_rewards']) + 1),
        'recompensa': metrics['episode_rewards'],
        'entregas': metrics['episode_deliveries'],
        'steps': metrics['episode_steps'],
        'taxa_sucesso': metrics['success_rates'],
        'colisoes': metrics['collisions'],
        'falha_r1': metrics['failures_r1'],
        'falha_r2': metrics['failures_r2']
    })
    df.to_csv(save_dir / 'metricas_treinamento_hatrpo.csv', index=False)
    print(f"📊 Métricas salvas em: {save_dir / 'metricas_treinamento_hatrpo.csv'}")


# ==================== FUNÇÃO PRINCIPAL ====================
def run_hatrpo_training(num_episodes=3000):
    config = HATRPOConfigOptimized()

    print("=" * 80)
    print("🤖 TREINAMENTO HATRPO OTIMIZADO - 3000 EPISÓDIOS")
    print("=" * 80)
    print(f"\n📋 CONFIGURAÇÃO OTIMIZADA:")
    print(f"   • Modelo: HATRPO com redes melhoradas")
    print(f"   • Dispositivo: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
    print(f"   • Total de episódios: {num_episodes}")
    print(f"   • Actor LR: {config.ACTOR_LR}")
    print(f"   • Critic LR: {config.CRITIC_LR}")
    print(f"   • Hidden Dim: {config.HIDDEN_DIM}")
    print(f"   • Num Layers: {config.NUM_LAYERS}")
    print(f"   • Entropy Coef: {config.ENTROPY_COEF}")
    print(f"   • Pickup Reward: {config.PICKUP_REWARD}")
    print(f"   • Drop Reward: {config.DROP_REWARD}")
    print("=" * 80)

    env = WarehouseEnv(config=config)
    sample_all_states, _ = env.reset()
    state_dim = sample_all_states.shape[1]
    action_dim = 6
    n_agents = 2
    global_state_dim = env.global_state_dim
    env.close()

    print(f"\n📊 DIMENSÕES:")
    print(f"   • Estado individual: {state_dim}")
    print(f"   • Estado global: {global_state_dim}")
    print(f"   • Ações por agente: {action_dim}")
    print(f"   • Robôs: {n_agents}")
    print(f"   • Caixas: 8")

    agents = [HATRPOAgentOptimized(i, state_dim, action_dim, global_state_dim, config)
              for i in range(n_agents)]

    critic = CentralizedCriticOptimized(global_state_dim, config)

    metrics = {
        'episode_rewards': [],
        'episode_deliveries': [],
        'episode_steps': [],
        'success_rates': [],
        'collisions': [],
        'failures_r1': [],
        'failures_r2': []
    }

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_dir = Path(f"hatrpo_optimized_results_{timestamp}")
    results_dir.mkdir(exist_ok=True)
    print(f"\n📁 Diretório de resultados: {results_dir.absolute()}")

    total_start_time = time.time()

    for episode in range(num_episodes):
        all_states, global_state = env.reset()
        episode_reward = 0
        episode_collisions = 0
        episode_step = 0

        # Coletar trajetória
        trajectory_buffer = TrajectoryBuffer()

        for step in range(config.MAX_STEPS):
            # Selecionar ações
            actions = []
            log_probs = []
            for agent_id, agent in enumerate(agents):
                action, log_prob = agent.select_action(all_states[agent_id], training=True)
                actions.append(action)
                log_probs.append(log_prob)

            # Obter valor do crítico
            value = critic.get_value(global_state)

            # Executar ações
            next_all_states, rewards, terminated, truncated, info, _, next_global_state = env.step(actions)

            # Armazenar na trajetória
            trajectory_buffer.push(all_states.flatten(), np.array(actions), sum(rewards),
                                  next_all_states.flatten(), terminated or truncated,
                                  np.array(log_probs), value, global_state, next_global_state)

            episode_reward += sum(rewards)
            episode_collisions = info['collisions']

            all_states = next_all_states
            global_state = next_global_state
            episode_step = step

            if terminated or truncated:
                break

        # Obter trajetória completa
        traj = trajectory_buffer.get_trajectory()

        if len(traj['states']) > 0:
            # Calcular advantages e returns
            values = critic.get_values(traj['global_states'])
            advantages, returns = critic.compute_gae(traj['rewards'], values, traj['dones'])

            # Normalizar advantages para estabilidade
            if len(advantages) > 1 and advantages.std() > 1e-8:
                advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

            # Atualizar crítico
            critic_loss = critic.update(traj['global_states'], returns)

            # Atualizar cada agente
            for agent_id, agent in enumerate(agents):
                # Preparar dados para o agente
                agent_states = traj['states'][:, agent_id * state_dim:(agent_id + 1) * state_dim]
                agent_actions = traj['actions'][:, agent_id]
                agent_old_log_probs = traj['log_probs'][:, agent_id]

                # Atualizar ator
                actor_loss = agent.update_actor(agent_states, agent_actions, advantages, agent_old_log_probs)

        metrics['episode_rewards'].append(episode_reward)
        metrics['episode_deliveries'].append(info['total_deliveries'])
        metrics['episode_steps'].append(episode_step + 1)
        metrics['success_rates'].append(info['success_rate'])
        metrics['collisions'].append(episode_collisions)
        metrics['failures_r1'].append(info['failures_r1'])
        metrics['failures_r2'].append(info['failures_r2'])

        if (episode + 1) % 100 == 0:
            recent_rewards = metrics['episode_rewards'][-100:]
            recent_deliveries = metrics['episode_deliveries'][-100:]
            epsilon = agents[0].get_epsilon()
            elapsed = time.time() - total_start_time

            print(f"Ep {episode+1:4d}/{num_episodes} | "
                  f"Reward: {np.mean(recent_rewards):7.2f} | "
                  f"Entregas: {np.mean(recent_deliveries):.2f}/8 | "
                  f"ε: {epsilon:.3f} | "
                  f"Tempo: {elapsed/60:.1f}min")

        if (episode + 1) % config.CLEAN_MEMORY_EVERY == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    total_time = (time.time() - total_start_time) / 60
    env.close()

    print(f"\n💾 SALVANDO RESULTADOS...")

    save_metrics_csv(metrics, results_dir)
    plot_individual_graphs(metrics, results_dir)

    models_dir = results_dir / "models"
    models_dir.mkdir(exist_ok=True)
    for agent in agents:
        agent.save_model(models_dir, "final")
    critic.save_model(models_dir, "final")

    final_deliveries = metrics['episode_deliveries'][-100:]
    final_rewards = metrics['episode_rewards'][-100:]

    report = f"""
    ========================================
    RELATÓRIO FINAL - HATRPO OTIMIZADO (3000 EPISÓDIOS)
    ========================================

    DATA: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
    DIRETÓRIO: {results_dir.absolute()}

    CONFIGURAÇÃO OTIMIZADA:
    - Total de episódios: {num_episodes}
    - Tempo total: {total_time:.1f} minutos
    - Actor LR: {config.ACTOR_LR}
    - Critic LR: {config.CRITIC_LR}
    - Hidden Dim: {config.HIDDEN_DIM}
    - Num Layers: {config.NUM_LAYERS}
    - Entropy Coef: {config.ENTROPY_COEF}
    - Pickup Reward: {config.PICKUP_REWARD}
    - Drop Reward: {config.DROP_REWARD}

    MÉTRICAS FINAIS (últimos 100 episódios):
    - Recompensa média: {np.mean(final_rewards):.2f}
    - Entregas médias: {np.mean(final_deliveries):.2f}/8
    - Taxa de sucesso final: {metrics['success_rates'][-1]:.1%}

    MELHORIAS IMPLEMENTADAS:
    - Redes neurais mais profundas (3 camadas)
    - Hidden Dim aumentado (256 → 512)
    - Recompensas de pickup/drop aumentadas
    - Penalidade de movimento reduzida
    - Taxa de falha reduzida (20% → 15%)
    - Learning rates otimizados
    - Entropy bonus para exploração
    - Gradient clipping
    - Target networks com soft update

    ========================================
    """

    with open(results_dir / "relatorio_final_hatrpo.txt", 'w', encoding='utf-8') as f:
        f.write(report)

    print(report)
    print(f"\n✅ TREINAMENTO HATRPO OTIMIZADO CONCLUÍDO!")
    print(f"📁 Resultados salvos em: {results_dir.absolute()}")

    return agents, metrics, results_dir


if __name__ == "__main__":
    NUM_EPISODES = 3000

    print("\n" + "=" * 80)
    print("🎮 INICIANDO TREINAMENTO HATRPO OTIMIZADO")
    print("=" * 80)
    print("\n⚠️ MELHORIAS IMPLEMENTADAS:")
    print("   • Redes neurais mais profundas (3 camadas + residual)")
    print("   • Hidden Dim aumentado (256 → 512)")
    print("   • Recompensas ajustadas (Pickup: 5→10, Drop: 25→50)")
    print("   • Penalidade de movimento reduzida (-0.01 → -0.005)")
    print("   • Taxa de falha reduzida (20% → 15%)")
    print("   • Learning rates otimizados (Actor: 1e-4→3e-4, Critic: 1e-3→3e-3)")
    print("   • Entropy bonus para exploração\n")

    try:
        agents, metrics, results_dir = run_hatrpo_training(num_episodes=NUM_EPISODES)

        print("\n✨ TREINAMENTO HATRPO OTIMIZADO CONCLUÍDO COM SUCESSO! ✨")

        # Mostrar estatísticas finais
        print("\n📊 RESULTADOS FINAIS:")
        print(f"   • Recompensa média (últimos 100): {np.mean(metrics['episode_rewards'][-100:]):.2f}")
        print(f"   • Entregas médias (últimos 100): {np.mean(metrics['episode_deliveries'][-100:]):.2f}/8")
        print(f"   • Taxa de sucesso final: {metrics['success_rates'][-1]:.1%}")

    except Exception as e:
        print(f"\n❌ ERRO DURANTE O TREINAMENTO: {e}")
        import traceback
        traceback.print_exc()


🎮 INICIANDO TREINAMENTO HATRPO OTIMIZADO

⚠️ MELHORIAS IMPLEMENTADAS:
   • Redes neurais mais profundas (3 camadas + residual)
   • Hidden Dim aumentado (256 → 512)
   • Recompensas ajustadas (Pickup: 5→10, Drop: 25→50)
   • Penalidade de movimento reduzida (-0.01 → -0.005)
   • Taxa de falha reduzida (20% → 15%)
   • Learning rates otimizados (Actor: 1e-4→3e-4, Critic: 1e-3→3e-3)
   • Entropy bonus para exploração

🤖 TREINAMENTO HATRPO OTIMIZADO - 3000 EPISÓDIOS

📋 CONFIGURAÇÃO OTIMIZADA:
   • Modelo: HATRPO com redes melhoradas
   • Dispositivo: CUDA
   • Total de episódios: 3000
   • Actor LR: 0.0003
   • Critic LR: 0.003
   • Hidden Dim: 512
   • Num Layers: 3
   • Entropy Coef: 0.01
   • Pickup Reward: 10.0
   • Drop Reward: 50.0

📊 DIMENSÕES:
   • Estado individual: 38
   • Estado global: 30
   • Ações por agente: 6
   • Robôs: 2
   • Caixas: 8
  Agente 0 usando dispositivo: cuda
  Agente 1 usando dispositivo: cuda

📁 Diretório de resultados: /content/hatrpo_optimized_results_20

In [ ]:
# =============================